In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
from bertviz import head_view
from captum.attr import LayerIntegratedGradients, visualization, LayerConductance
import torch
import torch.nn.functional as F
from lime.lime_text import LimeTextExplainer
import shap
import transformers

##### Choose sentences to be evaluated

In [ ]:
df_litlat = pd.read_csv("data\\best_predictions_litlat.csv")
# 2-10 tokens
# Correct and incorrect
# 1 example per class - 20 sentences
chosen_tuples = []
chosen_classes = set()
chosen_classes_2 = set()
for index, row in df_litlat.iterrows():
    text = row["Fragments"]
    id = (row["True_labels"], row["Predicted_labels"])
    id2 = row["True_labels"]
    # if len(text.split()) > 7 and len(text.split()) < 10:
    if len(text.split()) > 2 and len(text.split()) < 10:
        # first gather correctly classified true_lab == pred_lab
        if id in chosen_classes:
            continue
        elif row["True_labels"] == row["Predicted_labels"]:
            chosen_tuples.append((row["Fragments"], row["True_labels"], row["Predicted_labels"], True))
            chosen_classes.add(id)
        if id2 in chosen_classes_2:
            continue
        elif row["True_labels"] != row["Predicted_labels"]:
            chosen_tuples.append((row["Fragments"], row["True_labels"], row["Predicted_labels"], False))
            chosen_classes_2.add(id2)

# Manually adding missing classes, not to include too long sentences from other classes
technique_33 = ("Norime, kad lietuviai noriai kurtų šeimas, būtų laimingi, susilauktų kuo daugiau vaikų.",3,3, True)
technique_332 = ("Reikia dėti visas pastangas ir susigrąžinti emigravusius lietuvius, kurių yra daug visame pasaulyje.",3,3, True)
technique_72 = ("specialiai apmokytas ožys, naudojamas skerdyklose, mėsos kombinatuose ir KITUR.",7,2, True)
technique_99 = ("Nors tikrovė įmantriai retušuojama, lietuviai masiškai palieka tėvynę nepuoselėdami vilčių kada nors čia sugrįžti.",9,9, True)
chosen_tuples.append(technique_33)
chosen_tuples.append(technique_99)
chosen_tuples.append(technique_332)
chosen_tuples.append(technique_72)
print(chosen_classes)
print(chosen_classes_2)
print(len(chosen_tuples))
print(chosen_tuples)
df_chosen_samples = pd.DataFrame(data=chosen_tuples, columns=["Fragments", "True_labels", "Predicted_labels", "correct_classification"])

### Attention scores as explainability measure

##### Using BertViz

In [ ]:
model_ckp = "best_model" # here add the files from the chosen model
mod = AutoModelForSequenceClassification.from_pretrained(model_ckp).eval()
tok = AutoTokenizer.from_pretrained(model_ckp)
correct_df = df_chosen_samples.loc[df_chosen_samples["correct_classification"]]
incorrect_df = df_chosen_samples.loc[df_chosen_samples["correct_classification"] == False]

##### Correct classification

In [ ]:

for index, row in correct_df.iterrows():
    text = row["Fragments"]
    print("Propaganda technique:", row["True_labels"])
    print("Original fragment:", text)
    inputs = tok(text, return_tensors='pt')

    out = mod(**inputs, output_attentions=True)

    attention = out['attentions']  
    tokens = tok.convert_ids_to_tokens(inputs['input_ids'][0]) 
    # model_view(attention, tokens) 
    head_view(attention, tokens) 

##### Incorrect classification

In [ ]:

# incorrect_df

for index, row in incorrect_df.iterrows():
    text = row["Fragments"]
    print("Ture propaganda technique:", row["True_labels"], "Predicted:", row["Predicted_labels"])
    print("Original fragment:", text)
    inputs = tok(text, return_tensors='pt')

    out = mod(**inputs, output_attentions=True)

    attention = out['attentions']  
    tokens = tok.convert_ids_to_tokens(inputs['input_ids'][0]) 
    # model_view(attention, tokens) 
    head_view(attention, tokens) 

### Integrated Gradients

In [ ]:
def model_output(inputs):
    return mod(inputs)[0]


def create_input_and_baseline(text):
    max_length = 512
    baseline_token_id = tok.pad_token_id
    sep_token_id = tok.sep_token_id
    cls_token_id = tok.cls_token_id

    text_ids = tok.encode(text, max_length=max_length, truncation=True, add_special_tokens=False)
    input_ids = [cls_token_id] + text_ids + [sep_token_id]
    token_list = tok.convert_ids_to_tokens(input_ids)

    baseline_input_ids = [cls_token_id] + [baseline_token_id] * len(text_ids) + [sep_token_id]
    return torch.tensor([input_ids], device="cpu"), torch.tensor([baseline_input_ids], device="cpu"), token_list


def average_attribution(attributions):
    attributions = attributions.sum(dim=-1).squeeze(0) # squeeze discards dimension of 1
    attributions = attributions / torch.norm(attributions)
    return attributions


def visualize_attribution(text, model_input, true_class):
    lig = LayerIntegratedGradients(model_output, model_input)

    input_ids, baseline_input_ids, all_tokens = create_input_and_baseline(text)
    logits = mod(input_ids).logits
    pred_class = logits.argmax(dim=1).item() # Can use predicted or true class to see interpretation

    attributions, delta = lig.attribute(inputs=input_ids,
                                        baselines=baseline_input_ids,
                                        target=pred_class,
                                        return_convergence_delta=True,
                                        internal_batch_size=1)
    
    avg_attributions = average_attribution(attributions)

    score_viz = visualization.VisualizationDataRecord(
    word_attributions=avg_attributions,
    pred_prob = torch.max(mod(input_ids)[0]).detach().numpy(),
    true_class=true_class,
    pred_class=pred_class,
    attr_class=text,
    attr_score=avg_attributions.sum(),
    raw_input_ids=all_tokens,
    convergence_score=delta)

    visualization.visualize_text([score_viz])


##### Correct Classification

In [ ]:
model_input = mod.roberta.embeddings

for index, row in correct_df.iterrows():
    text = row["Fragments"]
    true_class = row["True_labels"]
    print("Propaganda technique:", true_class)
    print("Original fragment:", text)
    visualize_attribution(text, model_input, true_class)

##### Incorrect Classification

In [ ]:
for index, row in incorrect_df.iterrows():
    text = row["Fragments"]
    true_class = row["True_labels"]
    print("Ture propaganda technique:", true_class, "Predicted:", row["Predicted_labels"])
    print("Original fragment:", text)
    visualize_attribution(text, model_input, true_class)

### LIME
##### Correct Classification

In [ ]:
# class_names = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# class_names = ['Appeal To Authority', 'Doubt', 'Emotional Expression', 'Following Behind','Reductio Ad Hitlerum', 
#                'Repetition', 'Simplification', 'Uncertainty', 'Waving The Flag', 'Whataboutism Red Herring Straw Man']

class_names = ['Appeal to A.', 'Doubt', 'Emotional Exp.', 'Follow Behind','Hitlerum', 
               'Repetition', 'Simplification', 'Uncertainty', 'Waving Flag', 'Whataboutism']
# Maybe explain that the titles have been truncated to fit into the visualization

def predictor(texts):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tok(texts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = mod(**inputs)
        probas = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
    return probas

explainer = LimeTextExplainer(class_names=class_names, random_state=0)

In [ ]:
for index, row in correct_df.iterrows():
    text = row["Fragments"]
    true_class = row["True_labels"]
    print("Propaganda technique:", true_class)
    print("Original fragment:", text)
    exp = explainer.explain_instance(text, predictor, num_features=20, num_samples=200, labels=[true_class])
    exp.show_in_notebook(text=text)

##### Incorrect Classification

In [ ]:
for index, row in incorrect_df.iterrows():
    text = row["Fragments"]
    true_class = row["True_labels"]
    predicted_class = row["Predicted_labels"]
    print("True propaganda technique:", true_class, "Predicted:", row["Predicted_labels"])
    print("Original fragment:", text)
    print("Explainer of true class")
    exp = explainer.explain_instance(text, predictor, num_features=20, num_samples=200, labels=[true_class])
    exp.show_in_notebook(text=text)
    print("Explainer of predicted class")
    exp = explainer.explain_instance(text, predictor, num_features=20, num_samples=200, labels=[predicted_class])
    exp.show_in_notebook(text=text)


### SHAP

In [ ]:
class_names = ['Appeal To Authority', 'Doubt', 'Emotional Expression', 'Following Behind','Reductio Ad Hitlerum', 
               'Repetition', 'Simplification', 'Uncertainty', 'Waving The Flag', 'Whataboutism Red Herring Straw Man']

pred = transformers.pipeline(
    "text-classification",
    model=mod,
    tokenizer=tok,
    device=0,
    return_all_scores=True,
)

explainer = shap.Explainer(pred, output_names=class_names)
type(explainer)

##### Correct Classification

In [ ]:
for index, row in correct_df.iterrows():
    text = row["Fragments"]
    text_list = [text]
    true_class = row["True_labels"]
    print("True propaganda technique:", true_class)
    print("Original fragment:", text)
    
    shap_values = explainer(text_list)
    shap.plots.text(shap_values)

##### Incorrect Classification


In [ ]:
for index, row in incorrect_df.iterrows():
    text = row["Fragments"]
    text_list = [text]
    true_class = row["True_labels"]
    predicted_class = row["Predicted_labels"]
    print("True propaganda technique:", true_class, "Predicted:", row["Predicted_labels"])
    print("Original fragment:", text)
    shap_values = explainer(text_list)
    shap.plots.text(shap_values)


Additional examples

In [ ]:
technique_332 = ("Reikia dėti visas pastangas ir susigrąžinti emigravusius lietuvius, kurių yra daug visame pasaulyje.",3,3, True)
technique_72 = ("specialiai apmokytas ožys, naudojamas skerdyklose, mėsos kombinatuose ir KITUR.",7,2, True)

# for index, row in correct_df.iterrows():
#     text = row["Fragments"]
#     text_list = [text]
#     true_class = row["True_labels"]
#     print("True propaganda technique:", true_class)
#     print("Original fragment:", text)
    
shap_values = explainer([technique_332[0]])
shap.plots.text(shap_values)

shap_values = explainer([technique_72[0]])
shap.plots.text(shap_values)